# 模块 E 综合项目：数据分析 + 检索 + 可交付报告

使用本地订单 JSON 模拟小型 AI 经营项目：清洗订单数据、统计、图表、TF-IDF 检索和 Markdown 报告，不依赖外部 API。

**项目类型：**教师/自学参考答案 · 分步骤完整代码

## 提交要求

1. 生成订单 JSON 并转换为 DataFrame
2. 清洗金额与数量，计算 GMV、类别贡献
3. 绘制经营柱形图并保存 PNG
4. 用 TF-IDF 检索企业知识库并生成报告
5. 检查 CSV、PNG、Markdown 全部交付文件

## 0. 准备工作：执行以下代码生成本地素材

这个单元格可以直接运行。素材为合成测试数据，无需联网。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

out=Path("E_results");out.mkdir(exist_ok=True)
rng=np.random.default_rng(42)
categories=["文具","饮料","食品"]
rows=[]
for i in range(90):
    rows.append({"订单号":i+1,"品类":categories[i%3],"单价":str(int(rng.integers(5,50))),
                 "数量":int(rng.integers(1,5))})
(out/"E_orders.json").write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding="utf-8")
df=pd.read_json(out/"E_orders.json")
print("订单数:",len(df))

## TODO 1：清洗金额与数量，计算 GMV、类别贡献

运行参考实现，观察变量变化与输出。 

In [ ]:
df["单价"]=pd.to_numeric(df["单价"],errors="coerce")
df["数量"]=pd.to_numeric(df["数量"],errors="coerce")
df=df.dropna(subset=["单价","数量"])
df["销售额"]=df["单价"]*df["数量"]
report=df.groupby("品类",as_index=False)["销售额"].sum()
gmv=float(df["销售额"].sum())
report.to_csv(out/"E_category_sales.csv",index=False,encoding="utf-8-sig")
print("GMV:",gmv)

## TODO 2：绘制经营柱形图并保存 PNG

运行参考实现，观察变量变化与输出。 

In [ ]:
fig,ax=plt.subplots()
ax.bar(["A","B","C"],report["销售额"])
ax.set_xticks([0,1,2],["A","B","C"])
ax.set_title("Sales by category");ax.set_xlabel("Category");ax.set_ylabel("Sales")
fig.tight_layout();fig.savefig(out/"E_sales.png",dpi=150);plt.close(fig)

## TODO 3：用 TF-IDF 检索企业知识库并生成报告

运行参考实现，观察变量变化与输出。 

In [ ]:
knowledge=[
    "文具销售适合校园开学季促销",
    "饮料销售需要关注天气与保质期",
    "食品销售需要遵守储存与质量安全要求"
]
question="食品怎样做好安全管理？"
v=TfidfVectorizer(analyzer="char",ngram_range=(2,3))
X=v.fit_transform(knowledge)
score=cosine_similarity(v.transform([question]),X)[0]
evidence=knowledge[int(score.argmax())]
markdown=f"# AI 经营诊断练习\n\n- 订单数：{len(df)}\n- GMV：{gmv:.2f}\n- 问题：{question}\n- 检索依据：{evidence}\n\n> 此报告仅为模拟检索结果，并未调用大模型。\n"
(out/"E_report.md").write_text(markdown,encoding="utf-8")
print("检索依据:",evidence)

## TODO 4：检查 CSV、PNG、Markdown 全部交付文件

运行参考实现，观察变量变化与输出。 

In [ ]:
for name in ["E_orders.json","E_category_sales.csv","E_sales.png","E_report.md"]:
    assert (out/name).is_file() and (out/name).stat().st_size>0,name
print("E 模块全部交付文件核验通过")

## 交付核验

- `E_results/E_orders.json`
- `E_results/E_category_sales.csv`
- `E_results/E_sales.png`
- `E_results/E_report.md`

完成后核对所有文件是否存在并可打开。参考答案在 `solutions/E_project_solution.ipynb`。

In [ ]:
# 交付文件自检：仅检查文件是否存在且非空，不自动判断模型效果
from pathlib import Path
folder=Path("E_results")
required=["E_orders.json","E_category_sales.csv","E_sales.png","E_report.md"]
missing=[name for name in required if not (folder/name).is_file() or (folder/name).stat().st_size==0]
if missing:
    print("缺失或为空的文件：",missing)
else:
    print("全部交付文件已生成")
